# 📖 Master Data Dictionary (Sơ đồ từ điển dữ liệu)

Dưới đây là danh mục các bảng đã được join trong thư mục `data/interim/`.

| Tên bảng | Grain (Đơn vị dòng) | Key chính | Ý nghĩa nghiệp vụ |
| :--- | :--- | :--- | :--- |
| `orders_customers_merged` | Mỗi dòng là 1 đơn hàng | `order_id` | Thông tin khách hàng đã mua đơn đó |
| `items_orders_merged` | Mỗi dòng là 1 sản phẩm trong đơn | `order_item_id` | Chi tiết từng món hàng, giá, số lượng |
| `items_reviews_merged` | Mỗi dòng là 1 sản phẩm | `product_id` | Tổng hợp đánh giá khách hàng cho sản phẩm |
| `items_returns_merged` | Mỗi dòng là 1 món hàng trả lại | `order_item_id` | Lý do và trạng thái hoàn hàng |


In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

INTERIM_PATH = Path("../data/interim")

In [2]:
interim_files = sorted([f for f in os.listdir(INTERIM_PATH) if f.endswith(".csv")])
print(f"Tìm thấy {len(interim_files)} bảng trong thư mục interim.\n")

dfs = {}
for file in interim_files:
    file_path = INTERIM_PATH / file
    dfs[file] = pd.read_csv(file_path)

list(dfs.keys())

Tìm thấy 18 bảng trong thư mục interim.



C:\Windows\Temp\ipykernel_15380\2905979881.py:7: DtypeWarning: Columns (5,6,46,47,49,50,51,52,55,56,57,59,60,61,62) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[file] = pd.read_csv(file_path)
C:\Windows\Temp\ipykernel_15380\2905979881.py:7: DtypeWarning: Columns (5,6,7,8,10,11,12,13,16,17,18,20,21,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[file] = pd.read_csv(file_path)
C:\Windows\Temp\ipykernel_15380\2905979881.py:7: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[file] = pd.read_csv(file_path)
C:\Windows\Temp\ipykernel_15380\2905979881.py:7: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[file] = pd.read_csv(file_path)
C:\Windows\Temp\ipykernel_15380\2905979881.py:7: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[file] = pd.read_csv(f

['1_fact_order_item_enriched.csv',
 '2_fact_order_enriched.csv',
 '3_customer_360.csv',
 '4_product_360.csv',
 '5_daily_business_panel.csv',
 '6_product_month_panel.csv',
 '_audit_conclusion.csv',
 '_audit_data_dictionary.csv',
 '_null_audit_detail.csv',
 'items_double_promotions.csv',
 'items_orders_merged.csv',
 'items_products_merged.csv',
 'items_returns_merged.csv',
 'items_reviews_merged.csv',
 'orders_customers_merged.csv',
 'orders_geography_merged.csv',
 'orders_payments_merged.csv',
 'orders_shipments_final.csv']

In [3]:
def build_audit_overview(dfs: dict) -> pd.DataFrame:
    rows = []
    for file_name, df in dfs.items():
        n_rows, n_cols = df.shape
        total_cells = n_rows * n_cols if n_rows * n_cols > 0 else np.nan
        total_nulls = int(df.isna().sum().sum())
        duplicate_rows = int(df.duplicated().sum())

        rows.append({
            "file_name": file_name,
            "rows": n_rows,
            "cols": n_cols,
            "total_null_values": total_nulls,
            "null_density_pct": round((total_nulls / total_cells) * 100, 2) if pd.notna(total_cells) else np.nan,
            "duplicate_rows": duplicate_rows,
            "duplicate_row_pct": round((duplicate_rows / n_rows) * 100, 4) if n_rows > 0 else np.nan
        })

    return pd.DataFrame(rows).sort_values(["rows", "cols"], ascending=[False, False]).reset_index(drop=True)

audit_overview = build_audit_overview(dfs)
audit_overview

,file_name,rows,cols,total_null_values,null_density_pct,duplicate_rows,duplicate_row_pct
0,1_fact_order_item_enriched.csv,714669,84,18624346,31.02,0,0.0
1,items_double_promotions.csv,714669,26,12489084,67.21,0,0.0
2,items_orders_merged.csv,714669,14,1152816,11.52,0,0.0
3,items_products_merged.csv,714669,14,1152816,11.52,0,0.0
4,items_returns_merged.csv,714669,13,4526466,48.72,0,0.0
5,items_reviews_merged.csv,714669,12,4158396,48.49,0,0.0
6,2_fact_order_enriched.csv,646945,21,697332,5.13,0,0.0
7,orders_customers_merged.csv,646945,14,0,0.00,0,0.0
8,orders_geography_merged.csv,646945,11,0,0.00,0,0.0
9,orders_payments_merged.csv,646945,11,0,0.00,0,0.0


In [4]:
data_dictionary = pd.DataFrame([
    {
        "table_name": "orders_customers_merged.csv",
        "grain": "1 dòng = 1 đơn hàng",
        "primary_key": "order_id",
        "base_table": "orders",
        "joined_tables": "customers",
        "business_meaning": "Thông tin đơn hàng + hồ sơ khách hàng đặt đơn",
        "expected_structural_nulls": "gender, age_group, acquisition_channel có thể null"
    },
    {
        "table_name": "orders_payments_merged.csv",
        "grain": "1 dòng = 1 đơn hàng",
        "primary_key": "order_id",
        "base_table": "orders",
        "joined_tables": "payments",
        "business_meaning": "Thông tin đơn hàng + thanh toán",
        "expected_structural_nulls": "về lý thuyết không nên null nhiều vì orders-payments là 1:1"
    },
    {
        "table_name": "orders_geography_merged.csv",
        "grain": "1 dòng = 1 đơn hàng",
        "primary_key": "order_id",
        "base_table": "orders",
        "joined_tables": "geography",
        "business_meaning": "Thông tin đơn hàng + khu vực giao hàng",
        "expected_structural_nulls": "không nên null nhiều nếu zip join đúng"
    },
    {
        "table_name": "orders_shipments_final.csv",
        "grain": "1 dòng = 1 đơn hàng",
        "primary_key": "order_id",
        "base_table": "orders",
        "joined_tables": "shipments",
        "business_meaning": "Thông tin đơn hàng + vận chuyển",
        "expected_structural_nulls": "ship_date, delivery_date, shipping_fee có thể null với đơn chưa shipped/delivered/returned"
    },
    {
        "table_name": "items_orders_merged.csv",
        "grain": "1 dòng = 1 line item trong đơn",
        "primary_key": "order_id + product_id (+ dòng item nếu có)",
        "base_table": "order_items",
        "joined_tables": "orders",
        "business_meaning": "Chi tiết từng sản phẩm trong đơn + context của đơn",
        "expected_structural_nulls": "thường không nên null nhiều ở cột lõi"
    },
    {
        "table_name": "items_products_merged.csv",
        "grain": "1 dòng = 1 line item trong đơn",
        "primary_key": "order_id + product_id (+ dòng item nếu có)",
        "base_table": "order_items",
        "joined_tables": "products",
        "business_meaning": "Chi tiết line item + thông tin sản phẩm",
        "expected_structural_nulls": "không nên null ở thuộc tính sản phẩm nếu product_id join đúng"
    },
    {
        "table_name": "items_returns_merged.csv",
        "grain": "1 dòng = 1 line item trong đơn",
        "primary_key": "order_id + product_id (+ dòng item nếu có)",
        "base_table": "order_items",
        "joined_tables": "returns",
        "business_meaning": "Chi tiết line item + thông tin hoàn trả nếu có",
        "expected_structural_nulls": "return_* null là bình thường cho item không bị trả"
    },
    {
        "table_name": "items_reviews_merged.csv",
        "grain": "1 dòng = 1 line item trong đơn",
        "primary_key": "order_id + product_id (+ dòng item nếu có)",
        "base_table": "order_items",
        "joined_tables": "reviews",
        "business_meaning": "Chi tiết line item + đánh giá nếu có",
        "expected_structural_nulls": "review_* và rating null là bình thường cho item chưa được review"
    },
    {
        "table_name": "items_double_promotions.csv",
        "grain": "1 dòng = 1 line item trong đơn",
        "primary_key": "order_id + product_id (+ dòng item nếu có)",
        "base_table": "order_items",
        "joined_tables": "promotions (promo 1 + promo 2)",
        "business_meaning": "Theo dõi khuyến mãi áp vào từng line item",
        "expected_structural_nulls": "promo_id, promo_id_2 và thuộc tính promo có thể null"
    },
    {
        "table_name": "1_fact_order_item_enriched.csv",
        "grain": "1 dòng = 1 line item enriched",
        "primary_key": "order_id + product_id (+ dòng item nếu có)",
        "base_table": "order_items",
        "joined_tables": "orders, products, promotions, returns, reviews, ...",
        "business_meaning": "Bảng fact item-level cho EDA sâu",
        "expected_structural_nulls": "promo_*, return_*, review_* thường null nhiều là bình thường"
    },
    {
        "table_name": "2_fact_order_enriched.csv",
        "grain": "1 dòng = 1 đơn hàng enriched",
        "primary_key": "order_id",
        "base_table": "orders",
        "joined_tables": "customers, payments, geography, shipments, item aggregates, ...",
        "business_meaning": "Bảng fact order-level cho funnel, logistics, payment",
        "expected_structural_nulls": "shipment_* có thể null tùy order_status"
    },
    {
        "table_name": "3_customer_360.csv",
        "grain": "1 dòng = 1 khách hàng",
        "primary_key": "customer_id",
        "base_table": "customers",
        "joined_tables": "orders, payments, returns, reviews aggregates",
        "business_meaning": "Chân dung khách hàng 360",
        "expected_structural_nulls": "demographic fields có thể null"
    },
    {
        "table_name": "4_product_360.csv",
        "grain": "1 dòng = 1 sản phẩm",
        "primary_key": "product_id",
        "base_table": "products",
        "joined_tables": "sales/item/return/review/inventory aggregates",
        "business_meaning": "Chân dung sản phẩm 360",
        "expected_structural_nulls": "review/return aggregates có thể null hoặc 0"
    },
    {
        "table_name": "5_daily_business_panel.csv",
        "grain": "1 dòng = 1 ngày",
        "primary_key": "date",
        "base_table": "daily aggregated transactions",
        "joined_tables": "sales, web_traffic, order/payment/return aggregates",
        "business_meaning": "Panel ngày cho trend, seasonality, forecasting",
        "expected_structural_nulls": "không nên null ở biến cốt lõi theo ngày"
    },
    {
        "table_name": "6_product_month_panel.csv",
        "grain": "1 dòng = 1 sản phẩm x tháng",
        "primary_key": "product_id + year + month",
        "base_table": "inventory / monthly aggregates",
        "joined_tables": "inventory + item monthly aggregates",
        "business_meaning": "Panel tháng theo sản phẩm để phân tích tồn kho - nhu cầu",
        "expected_structural_nulls": "một số chỉ số monthly return/review có thể null"
    }
])

data_dictionary

,table_name,grain,primary_key,base_table,joined_tables,business_meaning,expected_structural_nulls
0,orders_customers_merged.csv,1 dòng = 1 đơn hàng,order_id,orders,customers,Thông tin đơn hàng + hồ sơ khách hàng đặt đơn,"gender, age_group, acquisition_channel có thể null"
1,orders_payments_merged.csv,1 dòng = 1 đơn hàng,order_id,orders,payments,Thông tin đơn hàng + thanh toán,về lý thuyết không nên null nhiều vì orders-payments là 1:1
2,orders_geography_merged.csv,1 dòng = 1 đơn hàng,order_id,orders,geography,Thông tin đơn hàng + khu vực giao hàng,không nên null nhiều nếu zip join đúng
3,orders_shipments_final.csv,1 dòng = 1 đơn hàng,order_id,orders,shipments,Thông tin đơn hàng + vận chuyển,"ship_date, delivery_date, shipping_fee có thể null với đơn chưa shipped/delivered/returned"
4,items_orders_merged.csv,1 dòng = 1 line item trong đơn,order_id + product_id (+ dòng item nếu có),order_items,orders,Chi tiết từng sản phẩm trong đơn + context của đơn,thường không nên null nhiều ở cột lõi
5,items_products_merged.csv,1 dòng = 1 line item trong đơn,order_id + product_id (+ dòng item nếu có),order_items,products,Chi tiết line item + thông tin sản phẩm,không nên null ở thuộc tính sản phẩm nếu product_id join đúng
6,items_returns_merged.csv,1 dòng = 1 line item trong đơn,order_id + product_id (+ dòng item nếu có),order_items,returns,Chi tiết line item + thông tin hoàn trả nếu có,return_* null là bình thường cho item không bị trả
7,items_reviews_merged.csv,1 dòng = 1 line item trong đơn,order_id + product_id (+ dòng item nếu có),order_items,reviews,Chi tiết line item + đánh giá nếu có,review_* và rating null là bình thường cho item chưa được review
8,items_double_promotions.csv,1 dòng = 1 line item trong đơn,order_id + product_id (+ dòng item nếu có),order_items,promotions (promo 1 + promo 2),Theo dõi khuyến mãi áp vào từng line item,"promo_id, promo_id_2 và thuộc tính promo có thể null"
9,1_fact_order_item_enriched.csv,1 dòng = 1 line item enriched,order_id + product_id (+ dòng item nếu có),order_items,"orders, products, promotions, returns, reviews, ...",Bảng fact item-level cho EDA sâu,"promo_*, return_*, review_* thường null nhiều là bình thường"


In [5]:
data_dictionary.to_csv(INTERIM_PATH / "_audit_data_dictionary.csv", index=False)

In [6]:
EXPECTED_KEYS = {
    "orders_customers_merged.csv": ["order_id"],
    "orders_payments_merged.csv": ["order_id"],
    "orders_geography_merged.csv": ["order_id"],
    "orders_shipments_final.csv": ["order_id"],
    "items_orders_merged.csv": ["order_id", "product_id", "unit_price"],
    "items_products_merged.csv": ["order_id", "product_id", "unit_price"],
    "items_returns_merged.csv": ["order_id", "product_id", "unit_price"],
    "items_reviews_merged.csv": ["order_id", "product_id", "unit_price"],
    "items_double_promotions.csv": ["order_id", "product_id", "unit_price"],
    "1_fact_order_item_enriched.csv": ["order_id", "product_id", "unit_price"],
    "2_fact_order_enriched.csv": ["order_id"],
    "3_customer_360.csv": ["customer_id"],
    "4_product_360.csv": ["product_id"],
    "5_daily_business_panel.csv": ["date"] if "date" in dfs.get("5_daily_business_panel.csv", pd.DataFrame()).columns else [],
    "6_product_month_panel.csv": ["product_id", "year", "month"],
}

In [7]:
def key_duplicate_check(df, keys):
    if not keys:
        return {
            "expected_key": None,
            "key_null_rows": np.nan,
            "duplicate_key_rows": np.nan
        }

    existing_keys = [k for k in keys if k in df.columns]
    if len(existing_keys) != len(keys):
        return {
            "expected_key": ", ".join(keys),
            "key_null_rows": "missing_key_columns",
            "duplicate_key_rows": "missing_key_columns"
        }

    key_null_rows = int(df[existing_keys].isna().any(axis=1).sum())
    duplicate_key_rows = int(df.duplicated(subset=existing_keys).sum())

    return {
        "expected_key": ", ".join(existing_keys),
        "key_null_rows": key_null_rows,
        "duplicate_key_rows": duplicate_key_rows
    }

In [8]:
def column_null_profile(df, file_name):
    out = pd.DataFrame({
        "file_name": file_name,
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "null_count": df.isna().sum().values,
        "null_pct": (df.isna().mean() * 100).round(2).values,
        "n_unique": df.nunique(dropna=True).values
    })
    return out.sort_values(["null_count", "column"], ascending=[False, True]).reset_index(drop=True)

null_profiles = []
for file_name, df in dfs.items():
    null_profiles.append(column_null_profile(df, file_name))

null_audit_detail = pd.concat(null_profiles, ignore_index=True)
null_audit_detail.head(30)

,file_name,column,dtype,null_count,null_pct,n_unique
0,1_fact_order_item_enriched.csv,applicable_category_p2,object,714463,99.97,1
1,1_fact_order_item_enriched.csv,discount_value_p2,float64,714463,99.97,1
2,1_fact_order_item_enriched.csv,end_date_p2,object,714463,99.97,2
3,1_fact_order_item_enriched.csv,min_order_value_p2,float64,714463,99.97,2
4,1_fact_order_item_enriched.csv,promo_channel_p2,object,714463,99.97,1
5,1_fact_order_item_enriched.csv,promo_id_2,object,714463,99.97,2
6,1_fact_order_item_enriched.csv,promo_id_p2,object,714463,99.97,2
7,1_fact_order_item_enriched.csv,promo_name_p2,object,714463,99.97,2
8,1_fact_order_item_enriched.csv,promo_type_p2,object,714463,99.97,1
9,1_fact_order_item_enriched.csv,stackable_flag_p2,float64,714463,99.97,1


In [9]:
null_audit_detail.to_csv(INTERIM_PATH / "_null_audit_detail.csv", index=False)

In [10]:
NULL_CLASS_RULES = {
    "orders_customers_merged.csv": {
        "structural_null": [],
        "expected_nullable": ["gender", "age_group", "acquisition_channel"],
        "unexpected_null": ["order_id", "customer_id", "order_date", "zip"]
    },
    "orders_payments_merged.csv": {
        "structural_null": [],
        "expected_nullable": [],
        "unexpected_null": ["order_id", "payment_method", "payment_value", "installments"]
    },
    "orders_geography_merged.csv": {
        "structural_null": [],
        "expected_nullable": [],
        "unexpected_null": ["order_id", "zip", "city", "region", "district"]
    },
    "orders_shipments_final.csv": {
        "structural_null": ["ship_date", "delivery_date", "shipping_fee"],
        "expected_nullable": [],
        "unexpected_null": ["order_id", "order_status"]
    },
    "items_orders_merged.csv": {
        "structural_null": [],
        "expected_nullable": [],
        "unexpected_null": ["order_id", "product_id", "quantity", "unit_price"]
    },
    "items_products_merged.csv": {
        "structural_null": [],
        "expected_nullable": [],
        "unexpected_null": ["order_id", "product_id", "product_name", "category", "segment", "price", "cogs"]
    },
    "items_returns_merged.csv": {
        "structural_null": ["return_id", "return_date", "return_reason", "return_quantity", "refund_amount"],
        "expected_nullable": [],
        "unexpected_null": ["order_id", "product_id"]
    },
    "items_reviews_merged.csv": {
        "structural_null": ["review_id", "review_date", "rating", "review_title"],
        "expected_nullable": [],
        "unexpected_null": ["order_id", "product_id"]
    },
    "items_double_promotions.csv": {
        "structural_null": ["promo_id", "promo_id_2"],
        "expected_nullable": [],
        "unexpected_null": ["order_id", "product_id"]
    },
    "1_fact_order_item_enriched.csv": {
        "structural_null": ["promo_id", "promo_id_2", "return_id", "review_id", "rating"],
        "expected_nullable": ["gender", "age_group", "acquisition_channel"],
        "unexpected_null": ["order_id", "product_id", "quantity", "unit_price"]
    },
    "2_fact_order_enriched.csv": {
        "structural_null": ["ship_date", "delivery_date", "shipping_fee"],
        "expected_nullable": ["gender", "age_group", "acquisition_channel"],
        "unexpected_null": ["order_id", "customer_id", "order_date"]
    },
    "3_customer_360.csv": {
        "structural_null": [],
        "expected_nullable": ["gender", "age_group", "acquisition_channel"],
        "unexpected_null": ["customer_id"]
    },
    "4_product_360.csv": {
        "structural_null": [],
        "expected_nullable": [],
        "unexpected_null": ["product_id", "product_name", "category", "segment", "price", "cogs"]
    },
    "5_daily_business_panel.csv": {
        "structural_null": [],
        "expected_nullable": [],
        "unexpected_null": ["date"]
    },
    "6_product_month_panel.csv": {
        "structural_null": [],
        "expected_nullable": [],
        "unexpected_null": ["product_id", "year", "month"]
    }
}

In [11]:
def classify_nulls_for_file(df, file_name, rules_dict):
    rules = rules_dict.get(file_name, {})
    structural = set(rules.get("structural_null", []))
    expected = set(rules.get("expected_nullable", []))
    unexpected = set(rules.get("unexpected_null", []))

    rows = []
    for col in df.columns:
        null_count = int(df[col].isna().sum())
        null_pct = round(df[col].isna().mean() * 100, 2)

        if col in structural:
            null_class = "structural_null"
        elif col in expected:
            null_class = "expected_nullable"
        elif col in unexpected:
            null_class = "unexpected_null"
        else:
            null_class = "unclassified"

        rows.append({
            "file_name": file_name,
            "column": col,
            "null_count": null_count,
            "null_pct": null_pct,
            "null_class": null_class
        })

    return pd.DataFrame(rows)

classified_nulls = []
for file_name, df in dfs.items():
    classified_nulls.append(classify_nulls_for_file(df, file_name, NULL_CLASS_RULES))

classified_nulls_df = pd.concat(classified_nulls, ignore_index=True)
classified_nulls_df.head(30)

,file_name,column,null_count,null_pct,null_class
0,1_fact_order_item_enriched.csv,order_id,0,0.00,unexpected_null
1,1_fact_order_item_enriched.csv,product_id,0,0.00,unexpected_null
2,1_fact_order_item_enriched.csv,quantity,0,0.00,unexpected_null
3,1_fact_order_item_enriched.csv,unit_price,0,0.00,unexpected_null
4,1_fact_order_item_enriched.csv,discount_amount,0,0.00,unclassified
5,1_fact_order_item_enriched.csv,promo_id_p1,438353,61.34,unclassified
6,1_fact_order_item_enriched.csv,promo_id_2,714463,99.97,structural_null
7,1_fact_order_item_enriched.csv,order_date,0,0.00,unclassified
8,1_fact_order_item_enriched.csv,customer_id,0,0.00,unclassified
9,1_fact_order_item_enriched.csv,zip_x,0,0.00,unclassified


In [12]:
def summarize_file_audit(file_name, df):
    key_info = key_duplicate_check(df, EXPECTED_KEYS.get(file_name, []))

    classified = classify_nulls_for_file(df, file_name, NULL_CLASS_RULES)

    unexpected_null_cols = classified[
        (classified["null_class"] == "unexpected_null") &
        (classified["null_count"] > 0)
    ]

    structural_null_cols = classified[
        (classified["null_class"] == "structural_null") &
        (classified["null_count"] > 0)
    ]

    dup_key = key_info["duplicate_key_rows"]

    # grain_check mềm hơn
    if dup_key == "missing_key_columns":
        grain_check = "WARN"
    elif pd.isna(dup_key):
        grain_check = "WARN"
    elif dup_key == 0:
        grain_check = "PASS"
    elif dup_key <= 20:
        grain_check = "WARN"
    else:
        grain_check = "FAIL"

    # null risk
    if len(unexpected_null_cols) == 0:
        null_risk = "PASS"
    elif len(unexpected_null_cols) <= 2:
        null_risk = "WARN"
    else:
        null_risk = "FAIL"

    # overall status
    if grain_check == "FAIL" or null_risk == "FAIL":
        overall_status = "FAIL"
    elif grain_check == "WARN" or null_risk == "WARN":
        overall_status = "WARN"
    else:
        overall_status = "PASS"

    notes = []
    if dup_key not in [0, np.nan, "missing_key_columns"] and not pd.isna(dup_key):
        notes.append(f"duplicate by expected grain = {dup_key}")
    if len(unexpected_null_cols) > 0:
        notes.append("unexpected null in: " + ", ".join(unexpected_null_cols["column"].tolist()[:8]))
    if len(structural_null_cols) > 0:
        notes.append("structural null present (likely normal)")

    return {
        "file_name": file_name,
        "rows": len(df),
        "cols": df.shape[1],
        "expected_key": key_info["expected_key"],
        "key_null_rows": key_info["key_null_rows"],
        "duplicate_key_rows": key_info["duplicate_key_rows"],
        "grain_check": grain_check,
        "unexpected_null_columns": len(unexpected_null_cols),
        "null_risk": null_risk,
        "overall_status": overall_status,
        "notes": " | ".join(notes)
    }

audit_conclusion = pd.DataFrame([
    summarize_file_audit(file_name, df) for file_name, df in dfs.items()
]).sort_values(["overall_status", "file_name"]).reset_index(drop=True)

audit_conclusion

,file_name,rows,cols,expected_key,key_null_rows,duplicate_key_rows,grain_check,unexpected_null_columns,null_risk,overall_status,notes
0,1_fact_order_item_enriched.csv,714669,84,"order_id, product_id, unit_price",0,0,PASS,0,PASS,PASS,structural null present (likely normal)
1,2_fact_order_enriched.csv,646945,21,order_id,0,0,PASS,0,PASS,PASS,structural null present (likely normal)
2,3_customer_360.csv,121930,27,customer_id,0,0,PASS,0,PASS,PASS,
3,4_product_360.csv,2412,20,product_id,0,0,PASS,0,PASS,PASS,
4,5_daily_business_panel.csv,3833,24,date,0,0,PASS,0,PASS,PASS,
5,items_double_promotions.csv,714669,26,"order_id, product_id, unit_price",0,0,PASS,0,PASS,PASS,structural null present (likely normal)
6,items_orders_merged.csv,714669,14,"order_id, product_id, unit_price",0,0,PASS,0,PASS,PASS,
7,items_products_merged.csv,714669,14,"order_id, product_id, unit_price",0,0,PASS,0,PASS,PASS,
8,items_returns_merged.csv,714669,13,"order_id, product_id, unit_price",0,0,PASS,0,PASS,PASS,structural null present (likely normal)
9,items_reviews_merged.csv,714669,12,"order_id, product_id, unit_price",0,0,PASS,0,PASS,PASS,structural null present (likely normal)


In [13]:
audit_conclusion.to_csv(INTERIM_PATH / "_audit_conclusion.csv", index=False)

In [14]:
if "orders_payments_merged.csv" in dfs:
    df = dfs["orders_payments_merged.csv"].copy()
    payment_cols = [c for c in ["payment_method", "payment_value", "installments"] if c in df.columns]

    if payment_cols:
        bad_payment_rows = df[df[payment_cols].isna().any(axis=1)]
        print("orders_payments_merged - rows thiếu payment info:", len(bad_payment_rows))
        display(bad_payment_rows.head())

orders_payments_merged - rows thiếu payment info: 0


,order_id,order_date,customer_id,zip,order_status,device_type,order_source,payment_value,installments,payment_method_match_flag,payment_method


In [15]:
if "orders_shipments_final.csv" in dfs:
    df = dfs["orders_shipments_final.csv"].copy()
    ship_cols = [c for c in ["ship_date", "delivery_date", "shipping_fee"] if c in df.columns]

    if "order_status" in df.columns and ship_cols:
        has_shipment = df[ship_cols].notna().any(axis=1)
        should_have = df["order_status"].isin(["shipped", "delivered", "returned"])

        violation_missing = df[should_have & (~has_shipment)]
        violation_unexpected = df[(~should_have) & has_shipment]

        print("Thiếu shipment info dù status cần có shipment:", len(violation_missing))
        print("Có shipment info dù status không nên có:", len(violation_unexpected))

        display(violation_missing.head())
        display(violation_unexpected.head())

Thiếu shipment info dù status cần có shipment: 564
Có shipment info dù status không nên có: 0


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,ship_date,delivery_date,shipping_fee
645849,832884,2022-12-23,7906,19540,delivered,credit_card,mobile,paid_search,NaN,NaN,NaN
645881,832929,2022-12-23,71155,35244,shipped,credit_card,desktop,paid_search,NaN,NaN,NaN
645899,832955,2022-12-23,36911,47906,delivered,bank_transfer,desktop,referral,NaN,NaN,NaN
645917,832978,2022-12-23,95160,67211,delivered,paypal,mobile,organic_search,NaN,NaN,NaN
645970,833043,2022-12-23,133114,94024,delivered,credit_card,desktop,organic_search,NaN,NaN,NaN


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,ship_date,delivery_date,shipping_fee


In [16]:
key_cols = ["order_id", "product_id"]

dup_mask = dfs["items_orders_merged.csv"].duplicated(subset=key_cols, keep=False)
dup_rows = dfs["items_orders_merged.csv"][dup_mask].sort_values(key_cols)

print("Số dòng nằm trong nhóm trùng key:", len(dup_rows))
display(dup_rows.head(50))

Số dòng nằm trong nhóm trùng key: 32


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
12233,14280,976,1,4019.47,0.00,NaN,NaN,2012-08-31,53117,36301,delivered,apple_pay,desktop,paid_search
12234,14280,976,2,3937.99,0.00,NaN,NaN,2012-08-31,53117,36301,delivered,apple_pay,desktop,paid_search
99645,113379,786,6,694.34,0.00,NaN,NaN,2013-09-02,138130,92040,delivered,paypal,mobile,organic_search
99646,113379,786,1,699.37,0.00,NaN,NaN,2013-09-02,138130,92040,delivered,paypal,mobile,organic_search
189239,215525,1859,5,1896.11,0.00,NaN,NaN,2014-08-28,21034,32082,delivered,credit_card,desktop,email_campaign
189240,215525,1859,5,1897.20,0.00,NaN,NaN,2014-08-28,21034,32082,delivered,credit_card,desktop,email_campaign
190291,216740,791,8,793.10,0.00,NaN,NaN,2014-09-01,139800,90247,delivered,paypal,desktop,social_media
190292,216740,791,5,825.52,0.00,NaN,NaN,2014-09-01,139800,90247,delivered,paypal,desktop,social_media
214311,243342,777,7,1181.17,1653.64,PROMO-0010,NaN,2015-01-01,49624,24549,delivered,paypal,mobile,referral
214312,243342,777,5,1147.21,1147.21,PROMO-0010,NaN,2015-01-01,49624,24549,delivered,paypal,mobile,referral


In [17]:
dup_summary = (
    dup_rows
    .groupby(key_cols)
    .size()
    .reset_index(name="n_rows")
    .sort_values("n_rows", ascending=False)
)

display(dup_summary)

,order_id,product_id,n_rows
0,14280,976,2
1,113379,786,2
2,215525,1859,2
3,216740,791,2
4,243342,777,2
5,265263,2394,2
6,397622,2045,2
7,398385,2016,2
8,428208,2090,2
9,530051,2219,2


In [18]:
cols_to_check = [
    "order_id", "product_id", "quantity", "unit_price",
    "discount_amount", "promo_id", "promo_id_2"
]

existing_cols = [c for c in cols_to_check if c in dfs["items_orders_merged.csv"].columns]

display(
    dup_rows[existing_cols].sort_values(["order_id", "product_id"])
)

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
12233,14280,976,1,4019.47,0.00,NaN,NaN
12234,14280,976,2,3937.99,0.00,NaN,NaN
99645,113379,786,6,694.34,0.00,NaN,NaN
99646,113379,786,1,699.37,0.00,NaN,NaN
189239,215525,1859,5,1896.11,0.00,NaN,NaN
189240,215525,1859,5,1897.20,0.00,NaN,NaN
190291,216740,791,8,793.10,0.00,NaN,NaN
190292,216740,791,5,825.52,0.00,NaN,NaN
214311,243342,777,7,1181.17,1653.64,PROMO-0010,NaN
214312,243342,777,5,1147.21,1147.21,PROMO-0010,NaN


In [19]:
candidate_keys = [
    ["order_id", "product_id"],
    ["order_id", "product_id", "unit_price"],
    ["order_id", "product_id", "unit_price", "discount_amount"],
    ["order_id", "product_id", "promo_id", "promo_id_2"],
    ["order_id", "product_id", "quantity", "unit_price", "discount_amount", "promo_id", "promo_id_2"]
]

df = dfs["items_orders_merged.csv"]

results = []
for keys in candidate_keys:
    existing = [k for k in keys if k in df.columns]
    dup_count = df.duplicated(subset=existing).sum()
    results.append({
        "candidate_key": ", ".join(existing),
        "duplicate_rows": dup_count
    })

pd.DataFrame(results)

,candidate_key,duplicate_rows
0,"order_id, product_id",16
1,"order_id, product_id, unit_price",0
2,"order_id, product_id, unit_price, discount_amount",0
3,"order_id, product_id, promo_id, promo_id_2",16
4,"order_id, product_id, quantity, unit_price, discount_amount, promo_id, promo_id_2",0
